In [120]:
q1="I just discovered the course, can I still join?"
q2="I just found out about the program, can I still enroll?"



In [121]:
#%pip install -U sentence-transformers

In [122]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [123]:
v1 = model.encode(q1)

In [124]:
v1.shape

(384,)

In [125]:
v2 = model.encode(q2)

In [126]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [127]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [128]:
v1.dot(dv)

np.float32(0.32332397)

In [129]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [130]:
v2.dot(dv)

np.float32(0.019730574)

In [131]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-29 18:49:36--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.8’

ingest.py.8         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-29 18:49:37 (53.3 MB/s) - ‘ingest.py.8’ saved [888/888]



In [132]:
from ingest import load_faq_data

documents = load_faq_data()

In [133]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [134]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [135]:
len(texts)

1350

In [136]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [137]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [138]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [139]:
import numpy as np
X = np.array(vectors)

In [140]:
scores = X.dot(v1)

In [141]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [142]:
documents[2]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [143]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [144]:
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [145]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [146]:
top5

array([  2, 625, 907, 538,   7])

In [147]:
top5 = np.argsort(-scores)[:5]



In [148]:
top5

array([  2, 625, 907, 538,   7])

In [149]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [150]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

In [151]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-29 18:50:45--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.7’

rag_helper.py.7     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-29 18:50:45 (40.7 MB/s) - ‘rag_helper.py.7’ saved [2134/2134]



In [152]:
from google import genai

client = genai.Client(
    vertexai=True,
    project="llm-zoomcamp-500505",
    location="us-central1",
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello"
)

print(response.text)

Hello!


In [153]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [154]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
print(len(documents))

1350


In [155]:
index = build_index(documents)

In [156]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=client,   # أو llm_client إذا استخدمت هذا الاسم
    model="gemini-2.5-flash",
)

In [ ]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

In [ ]:
class RAGVector(RAGBase):
    
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
)

In [ ]:
query = "I just discovered the course. Can I join now?"

results = assistant.search(query)

print(len(results))

5


In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [ ]:
vs_index.fit(vectors, documents)


ValueError: Index already contains documents. Use clear() to reset the index or add() to append documents.

In [ ]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [ ]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [ ]:
vs_index.close()


In [ ]:
query = "I just discovered the course. Can I join now?"

results = assistant.search(query)

prompt = assistant.build_prompt(query, results)

print(prompt)

QUESTION: I just discovered the course. Can I join now?

CONTEXT:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [ ]:
query = "I just discovered the course. Can I join now?"

assistant.rag(query)

'Yes, you can still join the course. You can start whenever you want as the videos and GitHub materials are available.\n\nHowever, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [ ]:
answer = assistant.llm(prompt)

print(answer)

Yes, you can join now. You can start whenever you want, as the videos and GitHub materials are available.

If you wish to receive a certificate, you need to submit your project while submissions are still being accepted. You can also start learning and submitting homework without formal registration, as registration is primarily to gauge interest.


In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,   # 👈 هذا هو الصحيح
)

In [ ]:
class RAGVector:
    def __init__(self, embedder, index, llm_client, model="gemini-2.5-flash"):
        self.embedder = embedder
        self.index = index
        self.llm_client = llm_client
        self.model = model

    def search(self, query, num_results=5):
        query_vec = self.embedder.encode(query).tolist()
        return self.index.search(query_vec, num_results=num_results)

    def build_context(self, results):
        return "\n".join(
            f"{r.get('text', '')}" for r in results
        )

    def answer(self, query):
        results = self.search(query)
        context = self.build_context(results)

        prompt = f"""
        Answer using only the context.

        Context:
        {context}

        Question:
        {query}
        """

        response = self.llm_client.models.generate_content(
            model=self.model,
            contents=prompt
        )

        return response.text

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client
)

In [ ]:
vector_assistant = RAGBase(
    index=vindex,
    llm_client=client
)

vector_assistant.rag(query)

TypeError: VectorSearch.search() got an unexpected keyword argument 'boost_dict'

In [ ]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)

TypeError: VectorSearch.search() got an unexpected keyword argument 'boost_dict'

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [ ]:
vs_index.fit(vectors, documents)

ValueError: Index already contains documents. Use clear() to reset the index or add() to append documents.

In [ ]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

NameError: name 'vs_index' is not defined

In [ ]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [ ]:
vs_index.close()